# Model Training & Evaluation — AI4I 2020 Predictive Maintenance Dataset

This notebook walks through the full model training pipeline:
- Preprocessing and train/test split
- Cross-validation of 3 models
- XGBoost hyperparameter tuning
- Final model evaluation with metrics and plots

In [1]:
import sys
sys.path.append('..')

from src.data_loader import load_raw_data
from src.preprocessor import split_and_preprocess
from src.trainer import train_and_evaluate_all, tune_xgboost, train_final_model
from src.evaluator import evaluate_model, plot_confusion_matrix, plot_roc_curve, plot_feature_importance, compare_models

# Load and preprocess
df = load_raw_data(filepath='../data/raw/ai4i2020.csv')
X_train, X_test, y_train, y_test, scaler, feature_names = split_and_preprocess(df)

Data loaded successfully: 10000 rows, 14 columns
Train size: 8000 rows
Test size:  2000 rows
Features:   ['Type', 'air_temp_K', 'process_temp_K', 'rotational_speed_rpm', 'torque_Nm', 'tool_wear_min', 'temp_difference', 'power']
Failure rate in train: 3.39%
Failure rate in test:  3.40%


In [2]:
# Train and cross-validate all base models
cv_results = train_and_evaluate_all(X_train, y_train)

Training: Logistic Regression...
  F1: 0.2732 | ROC-AUC: 0.8377
Training: Random Forest...
  F1: 0.7541 | ROC-AUC: 0.8097
Training: XGBoost...
  F1: 0.8008 | ROC-AUC: 0.8934


In [3]:
# Tune XGBoost and train final model
best_estimator, best_params, best_cv_f1 = tune_xgboost(X_train, y_train)
final_model = train_final_model(X_train, y_train, best_params)

Tuning XGBoost with GridSearchCV...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
Best CV F1: 0.8089
Training final model on full training set...
Final model trained.


In [4]:
# Evaluate all models and compare
all_metrics = []

for name, result in cv_results.items():
    result["model"].fit(X_train, y_train)
    metrics = evaluate_model(result["model"], X_test, y_test, model_name=name)
    all_metrics.append(metrics)
    plot_confusion_matrix(result["model"], X_test, y_test, model_name=name)
    plot_roc_curve(result["model"], X_test, y_test, model_name=name)

tuned_metrics = evaluate_model(final_model, X_test, y_test, model_name="XGBoost Tuned")
all_metrics.append(tuned_metrics)
plot_confusion_matrix(final_model, X_test, y_test, model_name="XGBoost Tuned")
plot_roc_curve(final_model, X_test, y_test, model_name="XGBoost Tuned")
plot_feature_importance(final_model, feature_names, model_name="XGBoost Tuned")

compare_models(all_metrics)


Evaluation: Logistic Regression
F1 Score:   0.2921
Precision:  0.1756
Recall:     0.8676
ROC-AUC:    0.9344

Classification Report:
              precision    recall  f1-score   support

  No Failure       0.99      0.86      0.92      1932
     Failure       0.18      0.87      0.29        68

    accuracy                           0.86      2000
   macro avg       0.59      0.86      0.61      2000
weighted avg       0.97      0.86      0.90      2000

Confusion matrix saved to notebooks\plots\confusion_matrix_Logistic_Regression.png
ROC curve saved to notebooks\plots\roc_curve_Logistic_Regression.png

Evaluation: Random Forest
F1 Score:   0.7826
Precision:  0.9574
Recall:     0.6618
ROC-AUC:    0.9652

Classification Report:
              precision    recall  f1-score   support

  No Failure       0.99      1.00      0.99      1932
     Failure       0.96      0.66      0.78        68

    accuracy                           0.99      2000
   macro avg       0.97      0.83      0.89

C:\Users\babaw\OneDrive\Desktop\work\Project 10 — Classical ML Pipeline\notebooks\..\src\evaluator.py:121: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=sorted_importances, y=sorted_features, palette="viridis")


Feature importance plot saved to notebooks\plots\feature_importance_XGBoost_Tuned.png

Model                           F1  Precision   Recall      AUC
Logistic Regression         0.2921     0.1756   0.8676   0.9344
Random Forest               0.7826     0.9574   0.6618   0.9652
XGBoost                     0.8088     0.8088   0.8088   0.9803
XGBoost Tuned               0.7801     0.7534   0.8088   0.9787
